# 02_ner_training.ipynb

## Setup

In [1]:
import os
import sys
from pathlib import Path

# Mount colab
if True:
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')

PROJECT_ROOT: Path = Path("/content/drive/MyDrive/python/projects/nlp_projects/med/biomedical-extraction-pipeline")
os.chdir(PROJECT_ROOT)

# Add the PROJECT_ROOT directory to the Python path to import modules
if str(PROJECT_ROOT) in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"ℹ️ Current Python version: {sys.version}")

ℹ️ Current Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [2]:
# Install reqs (NOTE: restart after this)
if True:
    # Install the required packages
    %pip install -r {"requirements/02_ner_training.txt"}
   
    # # Upgrade IPython to ensure autoreload works correctly
    # %pip install Ipython --upgrade
    # %load_ext autoreload
    # %autoreload 2

INFO: pip is looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 58.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: starlette
    Found existing installation: starlette 0.52.1
    Uninstalling starlette-0.52.1:
      Successfully uninstalled starlette-0.52.1
  Attempting uninstall: gradio-client
    F

In [2]:
# --- src ---
from src.config import load_configs
from src.utils import dynamic_module_reloader, seed_everything
# from src.data_utils ...

# --- other ---
import pandas as pd
import torch
from dotenv import load_dotenv
from pprint import pprint
from datasets import DatasetDict

MODULES_TO_RELOAD = [
    "src",
    "src.config",
    "src.utils",
    "src.data_utils",
    # "src.data_utils.processing"
]

dynamic_module_reloader(modules=MODULES_TO_RELOAD, verbose=1)
config = load_configs()
load_dotenv()
seed_everything(config.env.seed)
pprint(config.to_dict())

# Constants
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Reloaded: src
Reloaded: src.config
Reloaded: src.utils
✅ Seed set to 42
{'data': {'data_folder': PosixPath('data/documents'),
          'dataset_name': 'bigbio/bc5cdr'},
 'env': {'seed': 42},
 'model': {'ner_model_checkpoint': 'microsoft/BiomedNLP-PubMedBERT-base-uncased',
           'ner_model_name': 'BiomedNLP-PubMedBERT-base-uncased-ner-abstract-bc5cdr'},
 'retrieval': {}}
Using device: cpu


In [ ]:
!nvidia-smi

## Data

### Load the processed dataset (bigbio/BC5CDR)

In [3]:
# Load dataset from disk to verify
loaded_flattened_dataset = DatasetDict.load_from_disk("data/processed/bc5cdr_flattened_hf")
loaded_flattened_dataset

DatasetDict({
    train: Dataset({
        features: ['document_id', 'text', 'entities', 'relations'],
        num_rows: 500
    })
    test: Dataset({
        features: ['document_id', 'text', 'entities', 'relations'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['document_id', 'text', 'entities', 'relations'],
        num_rows: 500
    })
})

In [12]:
from transformers import AutoTokenizer

# 3. Instantiate your domain tokenizer
model_checkpoint = "dmis-lab/biobert-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [4]:
def tokenize_and_align_labels(examples):
    # Tokenize the text while keeping track of character spans
    tokenized_inputs = tokenizer(
        examples["text"], 
        truncation=True, 
        max_length=512, 
        return_offsets_mapping=True
    )
    
    labels = []
    # Loop through each document in the batch
    for i, labels_list in enumerate(examples["entities"]):
        doc_labels = [0] * len(tokenized_inputs["input_ids"][i])
        offset_mapping = tokenized_inputs["offset_mapping"][i]
        
        # Look at every true entity found in the data prep step
        for ent in labels_list:
            start_char, end_char = ent["offsets"]
            ent_type = ent["type"] # 'Chemical' or 'Disease'
            
            b_tag = 1 if ent_type == "Chemical" else 3
            i_tag = 2 if ent_type == "Chemical" else 4
            
            first_match = True
            for idx, (start_tok, end_tok) in enumerate(offset_mapping):
                # Ignore special tokens like [CLS] or [SEP]
                if start_tok == 0 and end_tok == 0:
                    doc_labels[idx] = -100 # PyTorch default to ignore cross-entropy loss
                    continue
                    
                # Check if the token falls within the character bounds of the entity
                if start_tok >= start_char and end_tok <= end_char and start_tok < end_tok:
                    if first_match:
                        doc_labels[idx] = b_tag
                        first_match = False
                    else:
                        doc_labels[idx] = i_tag
                        
        labels.append(doc_labels)
        
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
_ = loaded_flattened_dataset['train'].select(range(2)).map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Dataset({
    features: ['document_id', 'text', 'entities', 'relations', 'input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'labels'],
    num_rows: 2
})

In [22]:
_.to_pandas()

,document_id,text,entities,relations,input_ids,token_type_ids,attention_mask,offset_mapping,labels
0,227508,Naloxone reverses the antihypertensive effect ...,"[{'mesh_id': 'D009270', 'offsets': [0, 8], 'te...","[{'chem_mesh': 'D008750', 'dis_mesh': 'D007022'}]","[101, 11896, 2858, 21501, 1162, 7936, 1116, 11...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[0, 0], [0, 2], [2, 4], [4, 7], [7, 8], [9, 1...","[-100, 1, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,354896,Lidocaine-induced cardiac asystole. Intravenou...,"[{'mesh_id': 'D008012', 'offsets': [0, 9], 'te...","[{'chem_mesh': 'D008012', 'dis_mesh': 'D006323'}]","[101, 5255, 2572, 2599, 2042, 118, 10645, 1768...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[0, 0], [0, 2], [2, 4], [4, 6], [6, 9], [9, 1...","[-100, 1, 2, 2, 2, 0, 0, 3, 4, 4, 4, 4, 0, 0, ..."


In [ ]:
loaded_flattened_dataset['train'].map(tokenize_and_align_labels, batched=True)

NameError: name 'tokenizer' is not defined

In [ ]:
# Map this over your datasets
tokenized_train = train_dataset.map(tokenize_and_align_labels, batched=True)
tokenized_test = test_dataset.map(tokenize_and_align_labels, batched=True)